# display version of python used 

In [1]:
!py --version

Python 3.10.8


# import the libary

In [2]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# device used

In [3]:
paddle.device.get_device()

'cpu'

# loading image path

In [4]:
i=4
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/4.jpg'

# read img

In [5]:
img = cv2.imread(image_path)

# run paddleocr on img 

In [6]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # setting tread 
    os.environ["OMP_NUM_THREADS"] = "8"
    os.environ["MKL_NUM_THREADS"] = "8"
    # applying ocr on the image
    ocr = PaddleOCR(use_doc_orientation_classify=False, 
        use_doc_unwarping=False, 
        use_textline_orientation=False, 
        lang='en',
        device=paddle.device.get_device(),
        cpu_threads=8 
       )
    result = ocr.predict(img)

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


In [7]:
# result

# extracting the table from the img 

In [8]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
   
    # converting img to gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

    # splits the image and clahe 
    # C – Contrast
    # L – Limited
    # A – Adaptive
    # H – Histogram
    # E – Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_clahe = clahe.apply(gray)    
    # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
    # Better threshold (adaptive handles uneven lighting)
    thresh = cv2.adaptiveThreshold(
        gray_clahe, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 10
    )
     
    
    # --- Horizontal lines ---
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
    # connect broken horizontal lines
    horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
    # --- Vertical lines ---
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
    vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
    # connect broken vertical lines
    vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
    
    
    # Combine
    boxes = cv2.add(horizontal, vertical)
    # cv2.imwrite(f'./output/opencv/boxes_combine{i}.jpeg', boxes)
    
    # Final closing to join gaps
    kernel = np.ones((10,10), np.uint8)
    boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
    cv2.imwrite(f'./output/opencv/boxes_fill_gap{i}.jpeg', boxes)  

    # kernel = np.ones((2,2), np.uint8)
    # erode = cv2.erode(boxes, kernel, iterations=2)
    # cv2.imwrite(f'./output/opencv/erode{i}.jpeg', erode)
    
    # adding border
    h, w = img.shape[:2]
    border=cv2.rectangle(boxes, (0,0), (w,h), (255,255,255), 3)
    cv2.imwrite(f'./output/opencv/bordered{i}.jpeg', border)
    
    # Find contours
    contours, _ = cv2.findContours(border, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    output = img.copy()
    cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

# find and draw the counter

In [220]:
output = img.copy()

# for index,cnt in enumerate(contours):
#     x,y,w,h = cv2.boundingRect(cnt)
#     # filter noise
#     if w > 100 and h > 100:
cv2.drawContours(output, contours, -1, (0,255,0), 1)
cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

True

In [9]:
# contours

# Downscaling the paddle ocr border to small

#### ref code 

In [10]:
# ref code 
# new_poly = []
# for poly in result[0]["rec_polys"]:
#     # convert to float for calculation
#     poly = np.array(poly, dtype=np.float32)
#     cv2.polylines(output, [np.array(poly, dtype=np.int32)],isClosed=True, color=(255, 0, 0),thickness=2) 

#     x_min, y_min = np.min(poly, axis=0)
#     x_max, y_max = np.max(poly, axis=0)
#     w = x_max - x_min
#     h = y_max - y_min

#     # adaptive shrink
#     shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
#     shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
#     # print(poly)
    
#     # compute center
#     center_x = np.mean(poly[:, 0])
#     center_y = np.mean(poly[:, 1])

#     poly_box=[]
#     for (x,y) in poly:
#         x_new=int(center_x+(x-center_x)*shrink_factor_h)
#         y_new=int(center_y+(y-center_y)*shrink_factor_v)
#         poly_box.append([x_new,y_new])
#     new_poly.append(np.array(poly_box, dtype=np.int16))
#     cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=2) 
# cv2.imwrite(f'./output/opencv/scaling{i}.jpeg', output)

In [11]:
def downscaling(poly_list):
    poly_downscaling=[]
    for poly in poly_list:
        # convert to float for calculation
        poly = np.array(poly, dtype=np.float32)
        cv2.polylines(output, [np.array(poly, dtype=np.int32)],isClosed=True, color=(255, 0, 0),thickness=2) 
    
        x_min, y_min = np.min(poly, axis=0)
        x_max, y_max = np.max(poly, axis=0)
        w = x_max - x_min
        h = y_max - y_min
    
        # adaptive shrink
        shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
        shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
        # print(poly)
        
        # compute center
        center_x = np.mean(poly[:, 0])
        center_y = np.mean(poly[:, 1])
    
        poly_box=[]
        for (x,y) in poly:
            x_new=int(center_x+(x-center_x)*shrink_factor_h)
            y_new=int(center_y+(y-center_y)*shrink_factor_v)
            poly_box.append([x_new,y_new])
        # cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=2) 
        # cv2.imwrite(f'./output/opencv/scaling{i}.jpeg', output)
        poly_downscaling.append(np.array(poly_box, dtype=np.int16))
    return poly_downscaling


# make an array by combining the ploy points , text , and confident scores

In [12]:
downscaling_poly=downscaling(result[0]['dt_polys'])
ocr_result = [(poly , text , scores)
    for poly , text , scores  in 
        zip(downscaling_poly,
            result[0]['rec_texts'],
            result[0]['rec_scores'])]

In [13]:
# downscaling_poly[0].tolist()

In [14]:
# downscaling_poly[0]

# to get the overlapping percentage 

In [15]:
import numpy as np

def rectangle_overlap_percentage(rectA, rectB ):
    # """
    # rectA and rectB: list of 4 points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    # Can also be numpy arrays of shape (4,2)
    # Returns overlap % relative to rectA
    # """
    # Convert to numpy arrays
    rectA = np.array(rectA)
    rectB = np.array(rectB)
    
    # Get bounding box
    x_min_A, x_max_A = rectA[:,0].min(), rectA[:,0].max()
    y_min_A, y_max_A = rectA[:,1].min(), rectA[:,1].max()
    
    x_min_B, x_max_B = rectB[:,0].min(), rectB[:,0].max()
    y_min_B, y_max_B = rectB[:,1].min(), rectB[:,1].max()
    
    # Overlap width & height
    overlap_width = max(0, min(x_max_A, x_max_B) - max(x_min_A, x_min_B))
    overlap_height = max(0, min(y_max_A, y_max_B) - max(y_min_A, y_min_B))
    
    # Intersection area
    intersection_area = overlap_width * overlap_height
    
    # Area of rectangle A
    area_A = (x_max_A - x_min_A) * (y_max_A - y_min_A)
    
    # Overlap %
    overlap_percentage = (intersection_area / area_A) * 100 if area_A > 0 else 0

    return overlap_percentage

In [16]:
# contours[1]

In [17]:
# rectA = np.array((contours[1].reshape(-1,2)).tolist())
# rectA

In [18]:
# result[0]['rec_polys'][8].tolist()

In [19]:
# for ocr in ocr_result:
#     ocr[0]
# np.array(ocr[2])

In [20]:
def rectangle_overlap_percentage(paddleocr_box, counters):
    #rectA = paddleocr text box
    #rectB = counter of the table
    # rectA and rectB: list of 4 points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    #how much of a is in b
    # Convert to numpy arrays
    rectA_rectB_merge=[]
    # for counter_i in range(len(counters)):
    for counter_poly in counters:
        ocr_box=[]
        # for paddleocr_i in range(len(paddleocr_box)):
        for paddleocr_poly in paddleocr_box:
            # rectA_org = paddleocr_box[paddleocr_i][0].tolist()
            # rectB_org = counters[counter_i][:, 0, :].tolist()
            
            # rectA_org = paddleocr_poly.tolist()
            # rectB_org = counter_poly.tolist()
                    
            # rectA = np.array(rectA_org)
            # rectB = np.array(rectB_org)
            
            rectA = np.array(paddleocr_poly[0].tolist())
            rectB = np.array((counter_poly.reshape(-1,2)).tolist())
            
            # Get bounding box
            x_min_A, x_max_A = rectA[:,0].min(), rectA[:,0].max()
            y_min_A, y_max_A = rectA[:,1].min(), rectA[:,1].max()
            
            x_min_B, x_max_B = rectB[:,0].min(), rectB[:,0].max()
            y_min_B, y_max_B = rectB[:,1].min(), rectB[:,1].max()
            
            # Overlap width & height
            overlap_width = max(0, min(x_max_A, x_max_B) - max(x_min_A, x_min_B))
            overlap_height = max(0, min(y_max_A, y_max_B) - max(y_min_A, y_min_B))
            
            # Intersection area
            intersection_area = overlap_width * overlap_height
            
            # Area of rectangle A
            area_A = (x_max_A - x_min_A) * (y_max_A - y_min_A)
            
            # Overlap %
            overlap_percentage = (intersection_area / area_A) * 100 if area_A > 0 else 0

            #append if overlapping is more 
            if(overlap_percentage > 90 and paddleocr_poly[2] > 0.8):
                ocr_box.append(paddleocr_poly[0]) 
            # else:
            #     continue
        if (ocr_box):
            rectA_rectB_merge.append((ocr_box,[counter_poly]))
        # rectA_rectB_merge.append(ocr_box)
    return rectA_rectB_merge


In [21]:
ans=rectangle_overlap_percentage(ocr_result,contours)

In [22]:
# ans

In [23]:
len(contours)

29

In [24]:
len(ans)

22

In [25]:
# ans[2][1]

In [40]:
np.array(ans[3][1]).tolist()

[[[[960, 1547]],
  [[961, 1546]],
  [[1006, 1546]],
  [[1007, 1547]],
  [[1052, 1547]],
  [[1053, 1548]],
  [[1138, 1548]],
  [[1139, 1549]],
  [[1139, 1577]],
  [[1138, 1578]],
  [[961, 1578]],
  [[960, 1577]]]]

In [46]:
img_test = cv2.imread(f'./output/opencv/output{i}.jpeg')
# for cnt_no in range(len(ans)):
#     cv2.drawContours(img_test, ans[cnt_no][1], -1, (255,0,0), 3)
#     for poly_ocr_box in ans[cnt_no][0]:
#         cv2.polylines(img_test, [np.array(poly_ocr_box, dtype=np.int32)],isClosed=True, color=(255,0,0 ),thickness=3)
#     cv2.imwrite(f'./output/opencv/test{i}.jpeg', img_test)



cnt_no=9
cv2.drawContours(img_test, ans[cnt_no][1], -1, (255,0,0), 3)
for poly_ocr_box in ans[cnt_no][0]:
    cv2.polylines(img_test, [np.array(poly_ocr_box, dtype=np.int32)],isClosed=True, color=(255,0,0 ),thickness=3)
cv2.imwrite(f'./output/opencv/test{i}.jpeg', img_test)


True

In [96]:
# for i in range(len(contours)):
#     for j in range(len(result1[0]['rec_polys'])):
#         overlap = rectangle_overlap_percentage(result1[0]['rec_polys'][i].tolist(),contours[j][:, 0, :].tolist())
#         if(overlap > 50):
#             print(f"Overlap % of Rectangle A: {overlap:.2f}%")
#             cv2.drawContours(output, contours, j, (255,255,0), 2)
#             cv2.polylines(output, [np.array(result1[0]['rec_polys'][i], dtype=np.int32)],isClosed=True, color=(0, 0, 255),thickness=2)
# cv2.imwrite('./output/opencv/imagetry.jpeg', output)

# arranging the contours horizontally and vertically 

#### find the center of x and y

In [11]:
def get_center_from_rect(rect):
    x, y, w, h = rect
    cx = x + w / 2
    cy = y + h / 2
    return cx, cy

#### grouping counter vertically 

In [12]:
def group_contours_into_lines_h(contours, x_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][0])  # sort by cx

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cx
        _, prev_cx,_, _ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cx - prev_cx) < x_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[2])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

#### grouping counter horizontal

In [13]:
def group_contours_into_lines_v(contours, y_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][1])  # sort by cy

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cy
        _, _, prev_cy,_ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cy - prev_cy) < y_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[1])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

In [14]:
ans_h,ans_org_h = group_contours_into_lines_h(contours)
ans_v,ans_org_v = group_contours_into_lines_v(contours)

In [16]:
for i_cnt in range(len(ans_org_h)):
    cv2.drawContours(output, ans_org_h[i_cnt], -1, (0,255,0), 3)
    cv2.imwrite(f'./output/opencv/output{i_cnt}.jpeg', output)

# arranging the text box of paddleocr horizontally

In [17]:
def get_center(box):
    #[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    x_coords = [p[0] for p in box]
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4


def group_boxes_into_lines(boxes, y_threshold=20):
    # Step 1: compute centers
    box_centers = [(box, get_center(box),text,score) for box,text,score in boxes]

    # Step 2: sort by Y (top to bottom)
    box_centers.sort(key=lambda b: b[1][1])

    lines = []
    current_line = []

    for box, (cx, cy) ,text ,score in box_centers:
        if score >= 0.9:
            if not current_line:
                current_line.append((box, cx, cy, text, score))
                continue
    
            _, _, prev_cy, _, _ = current_line[-1]
    
            # Step 3: check if same line
            if abs(cy - prev_cy) < y_threshold:
                current_line.append((box, cx, cy, text, score))
            else:
                lines.append(current_line)
                current_line = [(box, cx, cy, text, score)]
        else:
            continue

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line by X (left to right)
    sorted_lines = []
    sorted_lines_text = []
    for line in lines:
        line.sort(key=lambda b: b[2])  # sort by center x
        sorted_lines.append([b[0] for b in line])
        sorted_lines_text.append([b[3] for b in line])

    return sorted_lines,sorted_lines_text

In [18]:
poly,text = group_boxes_into_lines(ocr_result)

In [19]:
poly

[[array([[58,  4],
         ...,
         [56, 44]], shape=(4, 2), dtype=int16),
  array([[552,  19],
         ...,
         [551,  54]], shape=(4, 2), dtype=int16),
  array([[1304,   34],
         ...,
         [1303,   74]], shape=(4, 2), dtype=int16),
  array([[1497,   40],
         ...,
         [1496,   76]], shape=(4, 2), dtype=int16),
  array([[55, 41],
         ...,
         [55, 81]], shape=(4, 2), dtype=int16),
  array([[1754,   45],
         ...,
         [1753,   77]], shape=(4, 2), dtype=int16),
  array([[1628,   46],
         ...,
         [1628,   82]], shape=(4, 2), dtype=int16)],
 [array([[ 58,  99],
         ...,
         [ 58, 135]], shape=(4, 2), dtype=int16),
  array([[108,  99],
         ...,
         [106, 135]], shape=(4, 2), dtype=int16)],
 [array([[1294,  128],
         ...,
         [1293,  170]], shape=(4, 2), dtype=int16),
  array([[1520,  133],
         ...,
         [1519,  166]], shape=(4, 2), dtype=int16),
  array([[1625,  134],
         ...,
         [

In [20]:
text

[['SI', 'Description of Goods', 'Quantity', 'Rate', 'No.', 'Amount', 'per'],
 ['1', 'Books'],
 ['5,000 Nos', '38.00', 'Nos', 'Record Book', '1,90,000.00'],
 ['Size-1/4th Pages-4+96'],
 ['Cover-3o0gsm Duplex Board'],
 ['With Multicolor Printing'],
 ['& Matt Lamination'],
 ['Inner 60gsm Maplitho'],
 ['Single Color Printing'],
 ['Finishing-Perfect Binding'],
 ['2', 'Books'],
 ['2,500 Nos', '52.00', 'Nos', '1,30,000.00', 'Record Book'],
 ['Size-1/4th Pages-4+144'],
 ['Cover 300gsm Dup', 'Board'],
 ['With Multicolor Printing'],
 ['& Matt Lamination'],
 ['Inner-60gsm Maplitho Single'],
 ['Color Printing'],
 ['Finishing-Perfect Binding'],
 ['3,20,000.00'],
 ['CGST (Output) 9%',
  '9',
  '%',
  '28,800.00',
  '415342',
  'SGST (Output) 9%',
  '9',
  '28,800.00'],
 ['Total', '7,500 Nos', '3,77,600.00', 'Amount Chargeable (in words)'],
 ['E. &O.E'],
 ['INR Three Lakh Seventy Seven Thousand Six Hundred Only'],
 ['HSN/SAC', 'Taxable', 'Central Tax', 'State Tax', 'Total']]

In [21]:
ocr_result

[(array([[58,  4],
         ...,
         [56, 44]], shape=(4, 2), dtype=int16),
  'SI',
  0.9975608587265015),
 (array([[552,  19],
         ...,
         [551,  54]], shape=(4, 2), dtype=int16),
  'Description of Goods',
  0.9885762929916382),
 (array([[55, 41],
         ...,
         [55, 81]], shape=(4, 2), dtype=int16),
  'No.',
  0.99658203125),
 (array([[1304,   34],
         ...,
         [1303,   74]], shape=(4, 2), dtype=int16),
  'Quantity',
  0.9999382495880127),
 (array([[1497,   40],
         ...,
         [1496,   76]], shape=(4, 2), dtype=int16),
  'Rate',
  0.9998124241828918),
 (array([[1628,   46],
         ...,
         [1628,   82]], shape=(4, 2), dtype=int16),
  'per',
  0.999910295009613),
 (array([[1754,   45],
         ...,
         [1753,   77]], shape=(4, 2), dtype=int16),
  'Amount',
  0.9999079704284668),
 (array([[ 58,  99],
         ...,
         [ 58, 135]], shape=(4, 2), dtype=int16),
  '1',
  0.9998181462287903),
 (array([[108,  99],
         ...,
    